# LOG05 | 미션 07 | Solution — 교관 전용

**학생에게 배포하지 마세요.** 이 Notebook은 근거 확인형 교관 해설, 모의 기록의 검토 예시와 평가 기준을 담습니다. 공개된 원자료가 없는 항목은 판정 불가로 남깁니다.

**미션:** 통신 재밍 및 기상 급변 모의 긴급 임무 중지 판단 스트레스 훈련

## 📦 실습자료 업로드 (모든 미션 공통 · 실습 전 필수)

아래 **「LOG05 실습자료 ZIP 업로드 및 압축 해제」** 코드셀을 **먼저 실행**하세요.

1. 교육기관에서 제공한 **`LOG05_실습프로젝트.zip`**을 준비합니다. 파일명이 다르더라도 LOG05 자료가 담긴 정상적인 ZIP 파일이면 업로드할 수 있습니다.
2. **Google Colab:** 셀 실행 후 나타나는 업로드 창에서 ZIP을 선택하면 자동으로 압축을 해제합니다. **로컬 Jupyter:** ZIP 선택 후 **「선택한 ZIP 압축 해제」** 버튼을 누릅니다.
3. 화면에서 **「압축 해제 완료」** 메시지와 실습자료 폴더 경로를 확인한 다음, 아래 미션 실습을 진행하세요.

**주의:** 업로드가 끝나지 않았다면 미션 실습을 시작하지 마세요. ZIP을 올렸다는 사실만으로 실습 결과나 교관 승인이 완료되는 것은 아닙니다. 아래 코드셀의 소스코드는 기본 숨김으로 설정되어 있습니다.


In [ ]:
# @title 📦 LOG05 실습자료 ZIP 업로드 및 압축 해제 { display-mode: "form" }
# 학습자 화면에는 결과와 업로드 인터페이스만 표시하도록 구성합니다.
from pathlib import Path
import os, zipfile, stat
from IPython.display import display, Markdown

LOG05_PROJECT_DIR = Path.cwd() / "LOG05_실습자료"
LOG05_PROJECT_READY = False

def _log05_extract_zip(zip_path):
    global LOG05_PROJECT_READY
    zip_path = Path(zip_path)
    if not zipfile.is_zipfile(zip_path):
        raise ValueError("유효한 ZIP 압축파일이 아닙니다. 실습자료 ZIP을 다시 선택해 주세요.")
    dst = LOG05_PROJECT_DIR.resolve()
    dst.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        entries = archive.infolist()
        if len(entries) > 20000 or sum(info.file_size for info in entries) > 1024**3:
            raise ValueError("압축파일의 항목 수 또는 해제 후 용량이 허용 범위를 초과합니다.")
        for info in entries:
            item = Path(info.filename.replace('\\', '/'))
            if item.is_absolute() or '..' in item.parts or not info.filename.strip():
                raise ValueError("안전하지 않은 ZIP 내부 경로가 포함되어 있습니다: " + info.filename)
            target = (dst / item).resolve()
            if not target.is_relative_to(dst):
                raise ValueError("실습 폴더 밖으로 벗어나는 파일 경로입니다.")
            mode = (info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise ValueError("심볼릭 링크가 포함된 ZIP은 허용하지 않습니다.")
        for info in entries:
            target = dst / Path(info.filename.replace('\\', '/'))
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as src, target.open('wb') as out:
                    import shutil
                    shutil.copyfileobj(src, out)
    files = [p for p in dst.rglob('*') if p.is_file()]
    if not files:
        raise ValueError("ZIP에 압축 해제할 실습자료 파일이 없습니다.")
    LOG05_PROJECT_READY = True
    print("✅ LOG05 실습자료 압축 해제 완료:", dst)
    print("📄 확인된 파일 수:", len(files))
    print("ℹ️ 실습자료 폴더:", dst)
    print("※ 업로드·압축 해제는 학습 수행 및 평가 완료를 뜻하지 않습니다.")
    return dst

display(Markdown("**실습 시작 전 필수:** 교육기관에서 제공한 `LOG05_실습프로젝트.zip`을 업로드하세요. 이름이 다르더라도 정상적인 `.zip` 파일을 선택할 수 있지만, **LOG05 실습자료가 포함된 ZIP**이어야 합니다. 업로드 및 압축 해제를 완료한 후 다음 실습 단계로 진행하세요."))
try:
    from google.colab import files as _colab_files
    print("📤 업로드 창에서 LOG05 실습자료 ZIP 파일을 선택하세요.")
    _uploaded = _colab_files.upload()
    _zip_files = [(name, content) for name, content in _uploaded.items() if name.lower().endswith('.zip')]
    if not _zip_files:
        print("⚠️ ZIP 파일이 업로드되지 않았습니다. 이 셀을 다시 실행하세요.")
    else:
        if len(_zip_files) > 1:
            print("ℹ️ 여러 ZIP 중 첫 번째 파일을 사용합니다:", _zip_files[0][0])
        _name, _data = _zip_files[0]
        _zip_path = Path.cwd() / Path(_name).name
        _zip_path.write_bytes(_data)
        _log05_extract_zip(_zip_path)
except ImportError:
    import ipywidgets as _widgets
    _file_input = _widgets.FileUpload(accept='.zip', multiple=False, description='ZIP 파일 선택')
    _extract_button = _widgets.Button(description='선택한 ZIP 압축 해제', button_style='success')
    _upload_message = _widgets.Output()
    def _get_uploaded_zip():
        value = _file_input.value
        if not value:
            raise ValueError('먼저 LOG05 실습자료 ZIP을 선택하세요.')
        item = next(iter(value.values())) if isinstance(value, dict) else value[0]
        name = item['name']
        data = item['content']
        if not name.lower().endswith('.zip'):
            raise ValueError('ZIP 파일만 선택할 수 있습니다.')
        return name, bytes(data)
    def _on_extract(_):
        with _upload_message:
            _upload_message.clear_output()
            try:
                name, data = _get_uploaded_zip()
                target = Path.cwd() / Path(name).name
                target.write_bytes(data)
                _log05_extract_zip(target)
            except Exception as exc:
                print('⚠️ 업로드 또는 압축 해제 실패:', exc)
    _extract_button.on_click(_on_extract)
    display(_widgets.VBox([_file_input, _extract_button, _upload_message]))
    print('로컬 Jupyter: ZIP을 선택한 뒤 「선택한 ZIP 압축 해제」 버튼을 누르세요.')


## 교관 해설 및 판단 예시

공개 순서대로 확인한 교육용 카드와 당시 학생 기록을 대조합니다. 공개 전 미래 카드 내용은 학생용에 노출하지 않습니다. 최초 판단 시간 2분, 별도의 모의 회신 시간, 복수 평가자 요구를 섞지 않습니다.

**예시 풀이의 기본 논리**

1. 원본 가상 자료에서 해당 사례·기록 번호, 기간과 분모를 찾습니다.
2. 관찰된 **사실**, AI가 제안한 **해석**, 학생이 내리는 **판단**, 부족한 **자료**를 별도로 적습니다.
3. 근거를 확인할 수 없는 주장에는 승인이나 성공 판정을 붙이지 않습니다.
4. 최초 답안과 수정 답안을 구분하며 AI 요청·답변 원문을 확인합니다.
5. 학생의 독립 수행과 실제 사람 평가 조건을 별도로 확인합니다.

**실제 제3자 평가/승인:** 제공된 문서만으로 자동 생성할 수 없습니다.

## 원자료에 근거한 교관용 작성 예시

### 예시 결과: 순차 공개 상황에서의 기록 양식

| 공개 순번 | 공개 시각 | 공개 당시 확인한 원문 | 최초 판단을 적은 시각 | 나중에 수정한 내용 | 구분 |
|---|---|---|---|---|---|
| 1 | 실제 입력 | 당시 제공된 카드만 기입 | 실제 입력 | 뒤 카드 공개 후에 별도 기입 | 미실시/모의/실제 |
| 2 이후 | 공개 뒤 입력 | 미래 카드 선공개 금지 | 실제 입력 | 시간과 이유를 구분 | 미실시/모의/실제 |

**교관 판정 예시:** 처음 판단 시간의 제한(2분)과 명령 전달/모의 회신 시각은 다른 항목입니다. AI 한 번의 응답을 서로 다른 실제 평가자 2명의 판단으로 계산하지 않습니다. 실제 카드 공개·평가 자료가 없다면 미실시로 남깁니다.

## 학생과 동일한 원본 자료

미션 7은 통제관이 상황 카드를 순서대로 공개하는 실습입니다. 아직 공개되지 않은 내용은 사전에 열람하지 마세요. 각 카드의 공개 시점, 당시 최초 판단, 사용한 자료, 수정 이유를 별도 기록하세요.

**제공되는 기본 자료:** 미션 7의 빈 작성 양식 및 활동 안내. 순차 공개 카드 원문은 통제관 전용이며 학생용 Notebook에 포함하지 않습니다.

## 활동별 교관 검토 해설

### 활동 1. 기준·역할·측정을 먼저 고정한다

**교관이 확인할 구체적인 수행:** 1. 단계 개요 목표: 통신 두절과 기상 급변이 겹치는 극한의 가상 스트레스 상황에서 공정하고 일관된 평가를 내리기 위해, 본격적인 시나리오 대응에 앞서 5영역 × 4등급 평가 체계 , 역할 및 장비별 상태 정의 , 그리고 사전 위임 확인 방법 을 사전에 명확히 고정합니다. 수행 방식: 주관적이고 임의적인 판단을 방지하기 위해 평가의 척도와 각 직책의 권한 범위를 먼저 정의하며, 훈련의 실효성을 높이기 위해 미래의 상황 카드를 AI에게 미리 노출하거나 주지 않는 원칙을 철저히 준수합니다. 2. 세부 수행 절차 5영역 × 4등급 평가 체계 구축: 의사결정의 속도, 데이터 근거의 타당성, 예외 상황 대처 능력, 통신 두절 대응, 비상 중지 적정성 등 핵심 5대 영역을 설정하고, 각 영역별로 4단계의 명확한 성취 등급 기준을 정의합니다. 역할 및 장비별 상태 정의: 지휘소 관제자, 원격 조종수, 승인권자 등 각 직책의 책임 영역과 수송 드론·자율주행 차량 등 무인 보급장비의 상태(정상, 주의, 위험, 통신 두절 등)를 명확히 구분합니다. 사전 위임 확인 방법 고정: 비상 상황 발생 시 현장 조종수에게 위임된 독자적인 중지 권한의 범위와 지휘소 승인이 반드시 필요한 조건을 사전에 점검하고 기록하는 방식을 확정합니다. AI 사전 정보 통제 (미래 카드 은닉): 훈련의 긴장감과 객관적 판단력을 유지하기 위해, 이후 단계에서 주어질 가상 시나리오(미래 카드)의 전개 내용을 사전에 AI에게 입력하거나 노출하지 않도록 통제합니다. 3. 주의 및 검증 사항 평가 기준이나 등급 산정이 모호하게 설정되면 후속 스트레스 훈련(2～3단계)에서 객관적인 채점이 불가능해지므로, 기준을 수치화하고 행동 지표 중심 형태로 구체화

**예시 답안의 근거:** 학생이 원본 자료 번호와 해당 원문·분모·조건을 연결했는지 확인합니다. 입력 자료가 없는 값은 '확인 필요'로 남겨야 합니다.

**평가 포인트:** 이 활동의 확인 근거 M07 활동1 — 기준·역할·측정을 먼저 고정한다: 5영역×4등급 과 역할·장비별 상태·사전 위임 확인 방법을 정한다. 미래 카드를AI에 주지 않는다. 확인 증거는 실제 제공 자료번호·본인 작성/질문·AI 회신·대조/수정의 해당 위치입니다. 계획만 작성했다면 실행 완료로 표시하지 않습니다. 단계 결과를 forms/LOG05_M07_forms.md와 최종 파일 rubric/log05_evaluation_rubric.md, LOG05_M07_forms.md, forms/LOG05_M07_forms.md에 연결합니다. 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 2. 현재 카드에 직접 판단·대화한다

**교관이 확인할 구체적인 수행:** 1. 단계 개요 목표: 통신 두절과 기상 급변이 복합적으로 발생하는 가상 스트레스 상황에서 6장의 실시간 상황 카드 를 순차적으로 부여받고, 각 카드별로 2분 이내에 신속하고 정확한 첫 판단 을 내리며, 이어지는 명령 하달, AI 모의 회신, 그리고 수신 미확인 상태를 체계적으로 기록합니다. 수행 방식: 고스트레스 전장 환경을 가정하여 시간 압박 속에서 직관적 추정이 아닌 객관적 데이터와 기 수립된 중지 조건을 바탕으로 즉각적인 의사결정을 내리고, 그 전과정을 누락 없이 문서화합니다. 2. 세부 수행 절차 6장의 실시간 상황 카드 순차 접수 및 검토: 단계별로 주어지는 6개의 비상 시나리오 카드를 하나씩 확인하고, 센서 경보, 통신 끊김, 기상 악화 지표를 파악합니다. 2분 이내 제1차 신속 판단 기록: 각 카드별로 주어진 제한 시간(2분) 내에 임무를 '계속'할 것인지, '즉시 중지(Abort)'할 것인지 결심하고 그 근거를 기록합니다. 명령 하달 및 AI 모의 회신 대응: 조종수 또는 현장 관제 채널을 통한 명령 하달 내용과, 통신 재밍 상태를 반영한 AI의 모의 회신(또는 응답 지연/두절 상태)을 별도의 로그로 분리하여 기록합니다. 수신 미확인 영역 분리 기록: 명령이 현장에 정상 도달했는지 확인되지 않는 '수신 미확인' 구간을 식별하고, 불확실한 상태에서의 추가 조치 계획을 명시합니다. 3. 주의 및 검증 사항 시간 압박(2분 제한)으로 인해 원문 데이터나 중지 조건을 소홀히 검토하고 성급하게 판단을 내리지 않도록 주의해야 합니다. AI 모의 회신의 내용이 실제 통신 두절 및 재밍 상황의 기술적 제약 범위 내에서 이루어졌는지 철저히 대조해야 합니다.

**예시 답안의 근거:** 학생이 원본 자료 번호와 해당 원문·분모·조건을 연결했는지 확인합니다. 입력 자료가 없는 값은 '확인 필요'로 남겨야 합니다.

**평가 포인트:** 이 활동의 확인 근거 M07 활동2 — 현재 카드에 직접 판단·대화한다: 6카드를 하나씩 받고 2분 이내 첫 판단을 기록한다. 이후 명령·AI 모의 회신·수신 미확인을 별도 기록한다. 확인 증거는 실제 제공 자료번호·본인 작성/질문·AI 회신·대조/수정의 해당 위치입니다. 계획만 작성했다면 실행 완료로 표시하지 않습니다. 단계 결과를 forms/LOG05_M07_forms.md와 최종 파일 rubric/log05_evaluation_rubric.md, LOG05_M07_forms.md, forms/LOG05_M07_forms.md에 연결합니다. 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 3. 독립 채점과 실제 로그를 검토한다

**교관이 확인할 구체적인 수행:** 1. 단계 개요 목표: 앞선 스트레스 훈련(2단계)에서 수행한 6개의 상황 카드별 판단 결과와 AI 모의 회신 로그를 바탕으로, 2명의 독립된 평가자가 각각 채점한 5영역 평가 결과의 일치율을 비교 하고, 실제 행동 기록과 AI 모의 반응 간의 차이를 엄정하게 구분 합니다. 수행 방식: 주관적 편향을 배제하고 평가의 객관성을 확보하기 위해 복수의 평가자가 독립적으로 검증하며, 제한 시간(2분) 초과 여부, 장비 식별 혼동, 중요 로그 누락 여부를 면밀히 대조합니다. 2. 세부 수행 절차 평가자 2명의 5영역 평가 일치율 비교: 의사결정 속도, 데이터 근거 타당성, 예외 상황 대처, 통신 두절 대응, 비상 중지 적정성의 5대 영역에서 두 명의 평가자가 부여한 등급과 점수의 일치 여부를 대조하고 교차 검증합니다. 실제 행동 기록과 AI 모의 회신 분리 검토: 학습자가 직접 내린 현장 판단 및 명령 하달 로그와, 통신 재밍 상황을 가정한 AI의 모의 응답(지연·두절 등) 기록을 명확히 구분하여 분석합니다. 오류 및 제약 조건 위반 사항 식별: 2분의 제한 시간을 초과한 의사결정 사례, 수송 드론과 자율주행 차량 간의 장비 혼동, 필수 보고 항목의 누락 여부를 철저히 확인합니다. 3. 주의 및 검증 사항 평가자 간 점수 편차가 클 경우, 평가 기준(1단계에서 고정된 척도)의 해석에 모호함이 있었는지 점검하고 객관적 근거를 바탕으로 조율해야 합니다. 형식적인 검토에 그치지 않고, 실제 로그에 드러난 허점이나 시간 초과 기록을 누락 없이 반영해야 신뢰할 수 있는 최종 루브릭 산출물( rubric/log05_evaluation_rubric.md )로 이어질 수 있습니다.

**예시 답안의 근거:** 학생이 원본 자료 번호와 해당 원문·분모·조건을 연결했는지 확인합니다. 입력 자료가 없는 값은 '확인 필요'로 남겨야 합니다.

**평가 포인트:** 이 활동의 확인 근거 M07 활동3 — 독립 채점과 실제 로그를 검토한다: 평가자 2명의 원문·5영역 일치율을 비교하고 실제/AI 모의를 구분한다. 장비 혼동·누락·2분 초과를 확인한다. 확인 증거는 실제 제공 자료번호·본인 작성/질문·AI 회신·대조/수정의 해당 위치입니다. 계획만 작성했다면 실행 완료로 표시하지 않습니다. 단계 결과를 forms/LOG05_M07_forms.md와 최종 파일 rubric/log05_evaluation_rubric.md, LOG05_M07_forms.md, forms/LOG05_M07_forms.md에 연결합니다. 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 4. 보완과 미실시를 전달한다

**교관이 확인할 구체적인 수행:** 1. 단계 개요 목표: 앞서 1～3단계를 통해 확립된 20칸 평가 루브릭 , 6개의 상황 카드별 대응 응답 , 명령 및 AI 모의 회신 로그 , 사전 위임 및 시간 기록 , 그리고 식별된 보완 사항 을 후속 미션인 미션 8(AAR 실패 사례 분석 및 중지 조건 민감도 재설계)과 미션 10(표준 체크리스트 완성)으로 정확하고 누락 없이 인계합니다. 수행 방식: 단절된 개별 실습으로 끝나지 않도록, 스트레스 훈련 과정에서 도출된 한계점, 미실시 항목, 그리고 보완 필요 사항들을 체계적으로 정리하여 후속 단계의 기초 데이터로 연계합니다. 2. 세부 수행 절차 20칸 루브릭 및 6응답 결과 정돈: 5영역 × 4등급 체계로 완성된 최종 평가 루브릭과 6장의 비상 상황 카드별 대응 결과를 후속 분석용 표준 서식에 맞춰 정리합니다. 명령/회신 로그 및 위임 적용 결과 이관: 현장 명령 하달 내역과 통신 재밍 상태에서의 AI 모의 회신 기록, 그리고 현장 조종수 위임 권한 적용 여부와 제한 시간 준수 여부를 명시합니다. 미실시 항목 및 보완 사항 명확화: 훈련 과정에서 물리적 제약이나 시간 부족으로 충분히 검증되지 못한 미실시 영역을 솔직하게 기록하고, 미션 8과 미션 10에서 개선해야 할 보완 방향을 제시합니다. 최종 결과 파일 연계 및 저장: 정리된 데이터를 지정된 결과 파일 경로( rubric/log05_evaluation_rubric.md , LOG05_M07_forms.md , forms/LOG05_M07_forms.md )에 빠짐없이 반영하고 저장합니다. 3. 주의 및 검증 사항 실제 훈련에서 누락되거나 미실시된 항목을 임의로 완료된 것처럼 허위로 작성하여 인계하지 않도록 주의해야 합니다. 본 단계에

**예시 답안의 근거:** 학생이 원본 자료 번호와 해당 원문·분모·조건을 연결했는지 확인합니다. 입력 자료가 없는 값은 '확인 필요'로 남겨야 합니다.

**평가 포인트:** 이 활동의 확인 근거 M07 활동4 — 보완과 미실시를 전달한다: 20칸 루브릭·6응답·명령/회신·위임 적용·시간·보완 사항을M8·M10에 넘긴다. 확인 증거는 실제 제공 자료번호·본인 작성/질문·AI 회신·대조/수정의 해당 위치입니다. 계획만 작성했다면 실행 완료로 표시하지 않습니다. 단계 결과를 forms/LOG05_M07_forms.md와 최종 파일 rubric/log05_evaluation_rubric.md, LOG05_M07_forms.md, forms/LOG05_M07_forms.md에 연결합니다. 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

## 원본 CMS 평가 기준

**평가 항목 1:** [정량적 평가] [필수] 긴급 중지 판단 훈련·루브릭 — 평가 영역 5개와 등급 4단계(탁월·우수·보통·미흡) 기준 정의율 100%인가, 카드 6장 모두에 판단과 명령 전달 기록이 있고 카드당 판단 2분 이내인가, 평가자 2명 채점 일치율 80% 이상인가, 위임 기준 적용 결과를 기록했는가. PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다. 목표미달·0기준·미측정은그대로표시합니다. 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.

**평가 항목 2:** [정성적 평가] Why/How/What: 본인 계획·실제 대화·근거 검토3건 이상·보완·재확인·최종 반영. PASS 증거 통신 두절과 기상 급변이 겹치는 가상 상황에서 임무를 계속할지 중지할지 빠르게 판단하는 스트레스 훈련을 하고, 판단 능력을 평가할 루브릭을 정의한다.의판단이제공자료·역할책임·교육용가정과연결됩니다. 사실/제안/미확인과실제/모의수행을구분합니다. 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.

**평가 항목 3:** [진행 추적 및 완료 기준] 실제 조원·두 평가자·2분 판단·20분 문서점검·3분 발표·실제 교관 확인을 AI 모의와 구분한다. 미실시를 PASS로 쓰지 않는다. 파일: rubric/log05_evaluation_rubric.md, LOG05_M07_forms.md, forms/LOG05_M07_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다. 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다. 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.

**교관 검토 기록:** 판정 [미확인 / 보완 필요 / PASS / FAIL]을 실제 기준에 따라 기입해야 하며, 자동 표시는 하지 않습니다.

In [ ]:
MISSION_JSON = '{"code": "LOG05_m07", "steps": [{"title": "기준·역할·측정을 먼저 고정한다"}, {"title": "현재 카드에 직접 판단·대화한다"}, {"title": "독립 채점과 실제 로그를 검토한다"}, {"title": "보완과 미실시를 전달한다"}]}'
# LOG05 학습 기록 도구 (표준 Python/Jupyter 환경용)
import json, zipfile, io, os, datetime
from pathlib import Path
from IPython.display import display, Markdown
try:
    import ipywidgets as widgets
except ImportError as exc:
    raise RuntimeError("ipywidgets가 필요합니다. 폐쇄망에서는 사전 설치된 로컬 Jupyter 또는 제공된 Markdown 양식을 사용하세요.") from exc

MISSION_DATA = json.loads(MISSION_JSON)
CODE = MISSION_DATA['code']
OUT_DIR = Path(CODE + '_results')
OUT_DIR.mkdir(exist_ok=True)
BACKUP = OUT_DIR / (CODE + '_record.json')
fields = {}
for no, item in enumerate(MISSION_DATA['steps'], 1):
    fields['activity_%02d' % no] = widgets.Textarea(
        value='', placeholder='내 판단, 확인한 자료 번호, AI 제안과 수정 이유를 직접 작성하세요.',
        description='활동 %s' % no, layout=widgets.Layout(width='98%',height='130px'),
        style={'description_width':'75px'})
for key, label in [('first_thought','AI에게 묻기 전 내 판단'), ('ai_request','실제로 보낸 AI 요청 전체'),
                   ('ai_response','실제로 받은 AI 답변 전체'), ('source_check','원자료 대조 3건과 수정 이유'),
                   ('final_result','최종 작성 결과와 근거'), ('limits','미실시/미측정/확인 필요 및 교관 검토 상태')]:
    fields[key] = widgets.Textarea(value='', description=label,
       layout=widgets.Layout(width='98%',height='125px'),style={'description_width':'190px'})
status = widgets.HTML(value='<b>아직 저장하지 않았습니다.</b> 작성과 교관 승인은 별개입니다.')
restore_input = widgets.Textarea(description='복원 JSON',placeholder='백업 JSON의 내용을 붙여넣으세요.',layout=widgets.Layout(width='98%',height='100px'),style={'description_width':'110px'})

def payload():
    return {'schema':'log05-student-v1','mission':CODE,'saved_at':datetime.datetime.now(datetime.timezone.utc).isoformat(),
            'answers':{key:w.value for key,w in fields.items()},
            'status':'작성 기록 (학습 PASS 또는 교관 승인 아님)'}

def save_record(_=None):
    obj=payload()
    if BACKUP.exists():
        stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
        BACKUP.rename(OUT_DIR / (CODE+'_previous_'+stamp+'.json'))
    BACKUP.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf8')
    status.value='<b>로컬 저장 완료:</b> '+str(BACKUP)+' (PASS 아님)'

def export_json(_=None):
    save_record()
    raw=BACKUP.read_bytes()
    try:
        from google.colab import files
        files.download(str(BACKUP))
    except ImportError:
        print('Colab이 아닙니다. 저장 위치:',BACKUP.resolve())

def restore_record(_=None):
    try:
        obj=json.loads(restore_input.value)
        if obj.get('mission')!=CODE or not isinstance(obj.get('answers'),dict):raise ValueError('다른 미션 또는 형식이 맞지 않는 백업입니다.')
        save_record()  # 복원 전 백업 저장
        for key,w in fields.items():w.value=str(obj['answers'].get(key,''))
        status.value='<b>복원 적용:</b> 기존 작성 내용은 이전 JSON으로 보관했습니다. 다시 저장하세요.'
    except Exception as exc:status.value='<b>복원 오류:</b> '+str(exc)

def export_review_zip(_=None):
    save_record()
    target=OUT_DIR / (CODE+'_review.zip')
    with zipfile.ZipFile(target,'w',compression=zipfile.ZIP_DEFLATED) as zz:
        zz.write(BACKUP,arcname=BACKUP.name)
        text='\n\n'.join('## '+key+'\n'+val for key,val in payload()['answers'].items())
        zz.writestr(CODE+'_work.md',text)
        zz.writestr('README.txt','검토용 보관 파일. 실제 평가 PASS 또는 교관 승인 자료가 아닙니다.')
    try:
        from google.colab import files
        files.download(str(target))
    except ImportError:
        print('Colab이 아닙니다. ZIP 위치:',target.resolve())

for i,item in enumerate(MISSION_DATA['steps'],1):
    display(Markdown('### 활동 %d. %s' % (i,item['title'])))
    display(fields['activity_%02d' % i])
for key,label in [('first_thought','최초 판단'),('ai_request','AI 요청 원문'),('ai_response','AI 답변 원문'),
 ('source_check','원자료 대조'),('final_result','최종 결과'),('limits','한계 및 실제 검토 상태')]:
    display(Markdown('### '+label));display(fields[key])
buttons=[]
for title,handler in [('작성 내용 저장',save_record),('작성 JSON 내려받기',export_json),('검토용 ZIP 내려받기',export_review_zip),('작성 JSON 복원',restore_record)]:
    b=widgets.Button(description=title,button_style='info');b.on_click(handler);buttons.append(b)
display(widgets.HBox(buttons[:2]));display(widgets.HBox(buttons[2:]));display(restore_input);display(status)
